# Week 8 — Visualisation & Putting It Together: Charts with matplotlib & seaborn
## Phase 2a Python | PORA Academy Cohort 7

By the end of this session, you will be able to:
- Create bar charts, line charts, and histograms with matplotlib
- Format charts (title, labels, figsize, colours)
- Use seaborn for cleaner charts
- Save charts to file

## Why visualisation matters

You have spent seven weeks pulling numbers out of the Olist dataset — order counts, revenue totals, review scores. But a table of numbers rarely convinces anyone. When Olist's operations team saw that **November 2017 spiked to 7,544 orders** (their Black Friday surge), it was a *chart* that made the spike jump off the page — not a spreadsheet cell.

In this session we turn the DataFrames you already know how to build into pictures. A single well-labelled line chart can show a whole year of order volume at a glance; a bar chart can rank Brazil's states so anyone can see **São Paulo dwarfing every other state at 41,746 orders**. To do this we need two libraries: `matplotlib` (the foundation) and `seaborn` (a friendlier layer on top).

By the end you will be able to take any grouped Olist result and turn it into a chart worth putting in front of a stakeholder.

### Setup — load the Olist data

Run this cell first. It mounts Google Drive and loads every Olist table into a DataFrame. From Week 3 onward we always start here, so the tables `orders`, `customers`, `order_items`, `products`, `reviews`, `payments`, and `sellers` are ready to use in every cell below.

In [ ]:
import pandas as pd
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define paths
olist_path = '/content/drive/MyDrive/olist-data'

# Load all datasets
orders = pd.read_csv(os.path.join(olist_path, 'olist_orders_dataset.csv'))
customers = pd.read_csv(os.path.join(olist_path, 'olist_customers_dataset.csv'))
order_items = pd.read_csv(os.path.join(olist_path, 'olist_order_items_dataset.csv'))
products = pd.read_csv(os.path.join(olist_path, 'olist_products_dataset.csv'))
reviews = pd.read_csv(os.path.join(olist_path, 'olist_order_reviews_dataset.csv'))
payments = pd.read_csv(os.path.join(olist_path, 'olist_order_payments_dataset.csv'))
sellers = pd.read_csv(os.path.join(olist_path, 'olist_sellers_dataset.csv'))
product_translations = pd.read_csv(os.path.join(olist_path, 'product_category_name_translation.csv'))

# Verify datasets loaded
print(f"✓ Loaded {len(orders):,} orders")       # expected: 99,441 orders
print(f"✓ Loaded {len(customers):,} customers")  # expected: 99,441 customers
print(f"✓ Loaded {len(reviews):,} reviews")      # expected: 99,224 reviews
print(f"✓ Loaded {len(payments):,} payments")    # expected: 103,886 payments

## 1. matplotlib foundations — line and bar charts

`matplotlib` is the original Python plotting library, and almost every other charting tool is built on top of it. Think of it like a blank canvas and a set of brushes: you call `plt.figure()` to size the canvas, then a drawing function like `plt.plot()` (for lines) or `plt.bar()` (for bars), then you decorate it with a title and axis labels, and finally `plt.show()` hangs it on the wall.

The workflow is always the same three moves — **size, draw, label** — no matter what kind of chart you want. A line chart is the natural choice when the x-axis is *time* (months across a year), because the connecting line shows the trend of rising and falling. A bar chart is the choice when the x-axis is *categories* (order statuses, states), because separate bars invite the eye to compare heights.

Below we build a line chart of monthly orders for 2017 and a bar chart of order statuses. Watch for the November spike — that is Black Friday landing in the data.

In [ ]:
import matplotlib.pyplot as plt

# --- Line chart: monthly orders across 2017 ---
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders_2017 = orders[orders['order_purchase_timestamp'].dt.year == 2017].copy()
orders_2017['month'] = orders_2017['order_purchase_timestamp'].dt.to_period('M')
monthly = orders_2017.groupby('month')['order_id'].count()

print(f"November 2017 orders: {monthly.loc['2017-11']:,}")  # expected: 7,544

plt.figure(figsize=(12, 5))                       # size the canvas
plt.plot(monthly.index.astype(str), monthly.values,
         marker='o', linewidth=2, color='steelblue')   # draw the line
plt.title('Monthly Orders — Olist 2017', fontsize=14)  # label everything
plt.xlabel('Month')
plt.ylabel('Number of Orders')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# --- Bar chart: top 5 order statuses ---
status = orders['order_status'].value_counts().head(5)
print(status)   # expected: delivered dominates (96,478), then shipped, canceled, unavailable, invoiced

plt.figure(figsize=(8, 4))
plt.bar(status.index, status.values,
        color=['#2ecc71', '#3498db', '#e74c3c', '#f39c12', '#9b59b6'])
plt.title('Order Status Distribution')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 2. seaborn — cleaner charts with less code

`seaborn` sits on top of matplotlib and does two things for you: it applies attractive default styling, and it understands pandas DataFrames directly, so you can point it at a column instead of first computing counts by hand. Think of matplotlib as raw ingredients and seaborn as a meal-kit — the same food, but a lot of the prep is done for you.

A great example is `sns.countplot()`: hand it a DataFrame and a column, and it counts the values *and* draws the bars in one line. We will use it for the review-score distribution, where you should see the classic J-shape of online reviews — a huge pile of 5-star ratings and a smaller-but-notable bump of angry 1-star ratings.

For proportions rather than counts, a pie chart answers "what share of the whole?" We will use plain matplotlib's `plt.pie()` for payment methods, where **credit card should own about 73.9% of the pie**.

In [ ]:
import seaborn as sns

# --- seaborn countplot: review score distribution ---
plt.figure(figsize=(8, 4))
sns.countplot(data=reviews, x='review_score', palette='RdYlGn')
plt.title('Review Score Distribution — Olist')
plt.xlabel('Review Score')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

score_counts = reviews['review_score'].value_counts().sort_index()
print(f"5-star reviews: {score_counts.loc[5]:,}")  # expected: 57,328
print(f"1-star reviews: {score_counts.loc[1]:,}")  # expected: 11,424

# --- pie chart: payment method share ---
pay_counts = payments['payment_type'].value_counts()
pay_pct = payments['payment_type'].value_counts(normalize=True) * 100
print(f"Credit card share: {pay_pct.loc['credit_card']:.1f}%")  # expected: 73.9%
print(f"Boleto share: {pay_pct.loc['boleto']:.1f}%")            # expected: 19.0%

plt.figure(figsize=(7, 7))
plt.pie(pay_counts.values, labels=pay_counts.index, autopct='%1.1f%%',
        colors=['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6'])
plt.title('Payment Method Distribution')
plt.show()

## Using AI assistance (DeepSeek)

From Week 4 you have had DeepSeek available to help write and debug code, and charting is one of the places it shines — it is easy to forget the exact matplotlib incantation for rotating tick labels or adding a legend. But a chart that *looks* right can still be *wrong*, so the four-step protocol matters more than ever here:

1. **Understand first** — say in plain English what chart you want and why, before you prompt.
2. **Prompt with context** — give the DataFrame name, the exact column names, and the chart type.
3. **Validate the output** — check the chart against a verified number. If your review-score chart does not show a 5-star bar near **57,328**, the code is wrong, not the curriculum.
4. **Never copy blindly** — you must be able to explain every line, including each `plt.` call.

Here is a well-formed prompt you could give DeepSeek for one of today's charts:

> I have a pandas DataFrame called `reviews` with a column `review_score` (integers 1 to 5).
> Using seaborn, how do I draw a bar chart showing how many reviews got each score,
> with a title and axis labels?

The reliable answer is a single `sns.countplot(data=reviews, x='review_score')` call plus title/label lines — exactly what we wrote above. If DeepSeek instead reaches for `reviews.plot()` with a manual `value_counts()`, that still works, but check the counts match the verified values before trusting the picture. The cell below shows how to *validate* a DeepSeek-generated chart before you trust it.

In [ ]:
# DeepSeek suggested this alternative to sns.countplot — validate it before trusting it.
ai_counts = reviews['review_score'].value_counts().sort_index()

# STEP 3 of the protocol: check the numbers against verified values FIRST.
print(f"5-star count: {ai_counts.loc[5]:,}")  # expected: 57,328
print(f"1-star count: {ai_counts.loc[1]:,}")  # expected: 11,424

# Numbers check out -> now the chart is safe to draw.
plt.figure(figsize=(8, 4))
plt.bar(ai_counts.index, ai_counts.values, color='teal')
plt.title('Review Scores (validated DeepSeek version)')
plt.xlabel('Review Score')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## Going deeper — histograms and saving charts to file

Two things separate a throwaway chart from a deliverable. First, **histograms** answer a different question than bar charts: instead of counting categories, they bucket a *continuous* number into ranges to show its shape. Delivery time is a perfect case — every order took some number of days, and a histogram reveals whether most deliveries cluster tight or spread into a long tail. The Olist delivery data has a **mean of 12.1 days** but a **median of 10.0 days**; the gap between them is the visual fingerprint of a right-skewed tail of slow deliveries.

Second, a chart you cannot share is useless. `plt.savefig()` writes the current figure to an image file — but only if you call it *before* `plt.show()`, because `show()` clears the canvas. Save at a decent `dpi` (dots per inch) so the image is crisp when someone drops it into a slide.

In [ ]:
# --- Histogram: delivery time in days ---
orders['order_delivered_customer_date'] = pd.to_datetime(
    orders['order_delivered_customer_date'], errors='coerce')
delivered = orders.dropna(subset=['order_delivered_customer_date']).copy()
delivered['delivery_days'] = (
    delivered['order_delivered_customer_date'] - delivered['order_purchase_timestamp']
).dt.days

print(f"Mean delivery time: {delivered['delivery_days'].mean():.1f} days")     # expected: 12.1 days
print(f"Median delivery time: {delivered['delivery_days'].median():.1f} days") # expected: 10.0 days

plt.figure(figsize=(10, 4))
plt.hist(delivered['delivery_days'], bins=40, range=(0, 60), color='mediumseagreen', edgecolor='white')
plt.title('Delivery Time Distribution — Olist')
plt.xlabel('Days from Purchase to Delivery')
plt.ylabel('Number of Orders')

# Save BEFORE show() — show() clears the canvas
plt.savefig('delivery_time_histogram.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved chart to delivery_time_histogram.png")  # expected: confirmation the file was written

### One more level — multiple charts in one figure with `plt.subplots()`

When you build a dashboard, you rarely want charts scattered one per figure — you want them arranged together so the story reads left to right. `plt.subplots(rows, cols)` gives you a grid of *axes* objects in a single figure. Instead of the `plt.plot()` / `plt.title()` style, you now call methods **on each axis**: `ax.plot(...)`, `ax.set_title(...)`, `ax.set_xlabel(...)`. This axis-based style is exactly what you will use on Thursday inside Streamlit's `st.pyplot()`, so it is worth meeting now.

In [ ]:
# Two charts side by side in one figure
fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(14, 4))

# Left axis: monthly 2017 line
ax_left.plot(monthly.index.astype(str), monthly.values, marker='o', color='steelblue')
ax_left.set_title('Monthly Orders — 2017')
ax_left.set_xlabel('Month')
ax_left.set_ylabel('Orders')
ax_left.tick_params(axis='x', rotation=45)

# Right axis: review score bars
ax_right.bar(score_counts.index, score_counts.values, color='indianred')
ax_right.set_title('Review Scores')
ax_right.set_xlabel('Score')
ax_right.set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f"Left peak (Nov 2017): {monthly.loc['2017-11']:,}")  # expected: 7,544
print(f"Right peak (5-star): {score_counts.loc[5]:,}")      # expected: 57,328

## Common mistakes

Charting has a handful of traps that catch every beginner. The most common one is calling `plt.savefig()` *after* `plt.show()` and ending up with a blank image file — because `show()` renders and then empties the canvas. Two other classics: forgetting `plt.figure()` so every chart piles onto the same axes, and passing a whole DataFrame to `plt.bar()` when it wants an x and a y. Here they are as comments so nothing actually crashes.

In [ ]:
# ── COMMON MISTAKE 1: saving after show() gives a blank file ────
# WRONG — show() clears the figure, so the saved PNG is empty:
# plt.plot(monthly.index.astype(str), monthly.values)
# plt.show()
# plt.savefig('chart.png')   # writes a blank image!

# CORRECT — save first, then show:
# plt.plot(monthly.index.astype(str), monthly.values)
# plt.savefig('chart.png')   # figure still on the canvas
# plt.show()

# ── COMMON MISTAKE 2: forgetting plt.figure() between charts ────
# WRONG — both plots land on the SAME axes, overlapping:
# plt.bar(status.index, status.values)
# plt.bar(pay_counts.index, pay_counts.values)   # drawn on top of the first!

# CORRECT — start a fresh figure for each chart:
# plt.figure(); plt.bar(status.index, status.values); plt.show()
# plt.figure(); plt.bar(pay_counts.index, pay_counts.values); plt.show()

# ── COMMON MISTAKE 3: axis labels are not optional ────
# A chart with no title or axis labels is unreadable to a stakeholder.
# ALWAYS add: plt.title(...), plt.xlabel(...), plt.ylabel(...)
print("Save before show, one figure() per chart, always label your axes.")
# expected: Save before show, one figure() per chart, always label your axes.

## Mini-challenge — payment value histogram

⏱ ~5 minutes

The `payments` DataFrame has a `payment_value` column (the amount paid, in Brazilian Reais). Build a histogram of payment values between R$0 and R$500 so we can see where most payments land.

**Your tasks:**
1. Create a figure sized `(10, 4)`.
2. Draw a histogram of `payments['payment_value']` with `bins=50` and `range=(0, 500)`.
3. Add a title and both axis labels.
4. Print the overall average payment value.

**Expected:** the printed average should be about **R$154.10**, and the histogram should be strongly right-skewed (most payments small, a long thin tail toward R$500).

In [ ]:
# Given — do not change:
# payments  (DataFrame with a 'payment_value' column, in R$)

# 1. Create the figure
# Your code here

# 2. Draw the histogram (bins=50, range=(0, 500))
# Your code here

# 3. Add title and axis labels
# Your code here

# plt.show()

# 4. Print the average payment value
# Your code here
# expected: Average payment value: R$154.10

## Part 3 — Group exercise (40 min)

Each group builds 3 charts using verified data. Compare your chart's values against the expected numbers below — if they don't match, your data wrangling is off, not the target.

**Chart 1:** Monthly order volume for 2017 (line chart).
Peak should show: **November 2017 = 7,544 orders**.

**Chart 2:** Top 10 states by total orders (horizontal bar chart).
**SP should be far ahead at 41,746**.

**Chart 3:** Review score distribution (bar chart).
Scores 1–5, count for each. Expected: **[11424, 3151, 8179, 19142, 57328]**.

In [ ]:
# ============ CHART 1: Monthly order volume for 2017 (line chart) ============
# Given: orders (with order_purchase_timestamp already datetime from earlier cells)
monthly_2017 = orders[orders['order_purchase_timestamp'].dt.year == 2017] \
    .groupby(orders['order_purchase_timestamp'].dt.to_period('M'))['order_id'].count()

# Your code here — plt.figure(), plt.plot(...), title, labels, show()
# Verify: monthly_2017.loc['2017-11'] should be 7,544


# ============ CHART 2: Top 10 states by total orders (horizontal bar) ============
# Merge orders with customers to get each order's state, then count.
orders_states = orders.merge(customers[['customer_id', 'customer_state']], on='customer_id')
top_states = orders_states['customer_state'].value_counts().head(10)

# Your code here — plt.figure(), plt.barh(...), title, labels, show()
# Verify: top_states.loc['SP'] should be 41,746


# ============ CHART 3: Review score distribution (bar chart) ============
review_dist = reviews['review_score'].value_counts().sort_index()

# Your code here — plt.figure(), plt.bar(...) or sns.countplot, title, labels, show()
# Verify: review_dist.tolist() should be [11424, 3151, 8179, 19142, 57328]

## Session summary

| Concept | Key Syntax | Example |
|---|---|---|
| Line chart | `plt.plot(x, y)` | `plt.plot(months, counts, marker='o')` |
| Bar chart | `plt.bar(x, y)` | `plt.bar(status.index, status.values)` |
| Horizontal bar | `plt.barh(x, y)` | `plt.barh(states.index, states.values)` |
| Histogram | `plt.hist(data, bins=)` | `plt.hist(delivery_days, bins=40)` |
| Pie chart | `plt.pie(vals, labels=)` | `plt.pie(pay.values, labels=pay.index)` |
| seaborn count | `sns.countplot(data=, x=)` | `sns.countplot(data=reviews, x='review_score')` |
| Format | `plt.title / xlabel / ylabel / figsize` | `plt.title('Orders'); plt.xlabel('Month')` |
| Save | `plt.savefig(path, dpi=)` | `plt.savefig('chart.png', dpi=150)` |

---
**Coming up Thursday**: Streamlit — turn these charts into an interactive web dashboard with year filters, KPI metric tiles, and live-updating plots, then learn how it deploys to Streamlit Community Cloud (the same workflow you'll use for the Phase 2c Capstone).